# Supply Chain Case Study
Companion notebook for the **Supply Chain Case Study** section of Chapter 11.

The value isn't a single lookup -- it's cross-system reasoning with clear exception rules, since ERP, WMS, TMS, supplier, and carrier systems don't always agree with each other.

This is the book's `handle_supply_exception`, unchanged:

In [1]:
def handle_supply_exception(ctx):
    if ctx["quality_hold"]:
        return "manual_inventory_review"
    if ctx["shipment_delayed"] and ctx["days_of_cover"] < 2:
        return "expedite_and_notify_planner"
    if ctx["supplier_missed_commit"]:
        return "open_supplier_followup"
    return "monitor"

In [2]:
contexts = [
    {"name": "healthy inventory, on-time supplier", "quality_hold": False, "shipment_delayed": False, "days_of_cover": 12, "supplier_missed_commit": False},
    {"name": "delayed shipment, low cover", "quality_hold": False, "shipment_delayed": True, "days_of_cover": 1, "supplier_missed_commit": False},
    {"name": "supplier missed commit date", "quality_hold": False, "shipment_delayed": False, "days_of_cover": 8, "supplier_missed_commit": True},
    {"name": "inventory looks fine but on quality hold", "quality_hold": True, "shipment_delayed": False, "days_of_cover": 10, "supplier_missed_commit": False},
]

for ctx in contexts:
    print(f"{handle_supply_exception(ctx):26s} <- {ctx['name']}")

monitor                    <- healthy inventory, on-time supplier
expedite_and_notify_planner <- delayed shipment, low cover
open_supplier_followup     <- supplier missed commit date
manual_inventory_review    <- inventory looks fine but on quality hold


The last case is the one the book calls out specifically: inventory *looks* healthy in ERP, but a quality hold blocks release -- `quality_hold` is checked first so the agent never suggests reallocating stock that can't actually ship.

**Cross-system disagreement**: if one system says "shipped" but the carrier feed shows no movement, mark the status as **uncertain**, not resolved -- don't let the more convenient answer win by default:

In [3]:
def reconcile_shipment_status(erp_status: str, carrier_has_movement: bool) -> dict:
    if erp_status == "shipped" and not carrier_has_movement:
        return {"status": "uncertain", "reason": "ERP says shipped but carrier feed shows no movement"}
    return {"status": erp_status, "reason": "systems agree"}


print(reconcile_shipment_status("shipped", carrier_has_movement=False))
print(reconcile_shipment_status("shipped", carrier_has_movement=True))

{'status': 'uncertain', 'reason': 'ERP says shipped but carrier feed shows no movement'}
{'status': 'shipped', 'reason': 'systems agree'}


Attach timestamps and a confidence flag any time systems disagree, so downstream production/order-impact reasoning knows to treat that fact as provisional -- not as settled ground truth.